# PPE Detection — GPU Training & Evaluation (Kaggle / Colab)

This notebook exists because the Cursor cloud agent environment that built this
repo has **no GPU and no `torch`/`ultralytics` installed** — everything that
needs real training or a Roboflow export has to run somewhere else. This
notebook is that "somewhere else." It's designed to run unmodified on either
**Kaggle** (free P100/T4, matching the original baseline author's setup) or
**Google Colab** (free T4).

**Before running:**

1. **Enable a GPU.**
   - Kaggle: Notebook Settings -> Accelerator -> GPU P100 (or T4 x2).
   - Colab: Runtime -> Change runtime type -> T4 GPU.
2. **Get a free Roboflow API key** (takes ~2 minutes, no payment needed):
   sign up at roboflow.com, then find your key at
   app.roboflow.com/settings/api.
   This *is* required even for these fully-public datasets - the Roboflow
   Python package refuses all access without one (confirmed while building
   this notebook: `ValueError: A valid API key must be provided`). It is not a
   paid-tier limitation.
3. Have this repo's `docs/PROJECT_PLAN.md` and `docs/DATASET_NOTES.md` handy for
   context on *why* each step below exists - this notebook implements
   Milestones M1-M3 from that plan.

**What this notebook does, end to end:**
1. Installs dependencies and clones this repo (for its merge/evaluation code).
2. Exports all 4 dataset sources (3 from Roboflow + 1 direct download, no key
   needed) and runs `scripts/build_unified_dataset.py` to merge them.
3. Trains a YOLOv8n baseline, then a YOLOv8s comparison run.
4. Runs this repo's own evaluation code (`src/evaluation/`) against the trained
   model to get calibration (ECE/Brier score) and safety-critical-class
   threshold optimization - not just Ultralytics' default metrics.
5. Shows you how to get the results back into the repo.

Cells are independent enough to re-run individually if one step fails partway
through (e.g. a dataset export times out) - you don't need to restart from
cell 1 every time.

## 1. Setup: install dependencies, clone the repo

In [ ]:
!pip install -q ultralytics roboflow pyyaml scikit-learn pillow

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected - check the accelerator setting mentioned above "
          "before continuing, training will be extremely slow on CPU.")

In [ ]:
import os

REPO_URL = "https://github.com/A-Kuo/Worker-Safety-PPE-Detection-Model.git"
REPO_DIR = "Worker-Safety-PPE-Detection-Model"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!git checkout cursor/ppe-detection-project-plan-7565  # or main, once merged
!git pull
!pwd

## 2. Roboflow API key

Enter your key below (it is never printed or committed anywhere - `getpass`
hides the input, and this notebook cell's output containing it should not be
saved/shared). On Kaggle you can alternatively use **Add-ons -> Secrets** and
read it from there instead of typing it each time; on Colab you can use
**Secrets** (the key icon in the left sidebar) with `google.colab.userdata`.

In [ ]:
import getpass

ROBOFLOW_API_KEY = getpass.getpass("Roboflow API key: ")
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
print("Key set (length {} chars).".format(len(ROBOFLOW_API_KEY)))

## 3. Export the 3 Roboflow-hosted sources + download the 1 no-key source

Project/version numbers below are confirmed in `docs/DATASET_NOTES.md`
(re-check there if a version has since changed upstream):

| source key | workspace | project | version | images |
|---|---|---|---|---|
| `construction_site_safety` | roboflow-universe-projects | construction-site-safety | 28 | 2,801 |
| `ppe_combined_model` | roboflow-universe-projects | personal-protective-equipment-combined-model | 4 | 44,002 |
| `hard_hat_universe` | universe-datasets | hard-hat-universe-0dy7t | 26 | 7,034 |

`ppe_combined_model` is large (44k images) - the export itself may take
several minutes. If you want to iterate faster first, comment it out below and
merge just the other 3 sources, then add it back in for a final run.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

ROBOFLOW_SOURCES = {
    "construction_site_safety": dict(workspace="roboflow-universe-projects", project="construction-site-safety", version=28),
    "ppe_combined_model": dict(workspace="roboflow-universe-projects", project="personal-protective-equipment-combined-model", version=4),
    "hard_hat_universe": dict(workspace="universe-datasets", project="hard-hat-universe-0dy7t", version=26),
}

DATASET_ROOT = "raw_datasets"
os.makedirs(DATASET_ROOT, exist_ok=True)
source_paths = {}

for key, cfg in ROBOFLOW_SOURCES.items():
    out_dir = os.path.join(DATASET_ROOT, key)
    if os.path.isdir(out_dir):
        print(f"[{key}] already exported at {out_dir}, skipping (delete the folder to re-export)")
        source_paths[key] = out_dir
        continue
    print(f"[{key}] exporting {cfg['project']} v{cfg['version']} ...")
    project = rf.workspace(cfg["workspace"]).project(cfg["project"])
    dataset = project.version(cfg["version"]).download("yolov8", location=out_dir)
    source_paths[key] = dataset.location
    print(f"[{key}] -> {dataset.location}")

print("\nExported sources:", source_paths)

In [ ]:
# Ultralytics Construction-PPE - no API key needed, downloaded directly.
ULTRALYTICS_PPE_DIR = os.path.join(DATASET_ROOT, "ultralytics_construction_ppe")
if not os.path.isdir(ULTRALYTICS_PPE_DIR):
    !mkdir -p {ULTRALYTICS_PPE_DIR}
    !curl -sL -o construction-ppe.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip
    !unzip -q construction-ppe.zip -d {ULTRALYTICS_PPE_DIR}
    !rm construction-ppe.zip
else:
    print(f"already downloaded at {ULTRALYTICS_PPE_DIR}, skipping")

source_paths["ultralytics_construction_ppe"] = ULTRALYTICS_PPE_DIR
print("All source paths:", source_paths)

## 4. Merge into the unified schema

This runs the exact same `scripts/build_unified_dataset.py` that was already
unit-tested and smoke-tested (against just the `ultralytics_construction_ppe`
source) in this repo's cloud-agent session - see `docs/DATA_DISTRIBUTION.md`
for that prior run's results. Running it here with all 4 sources for the first
time is the main thing this notebook unblocks.

If `src/data/label_schema.py` raises a `KeyError` about an unmapped class, it
means one of the Roboflow exports has a class name not yet accounted for
(`ppe_combined_model` and `hard_hat_universe` are marked
`status="pending_export"` in that file for exactly this reason) - open
`src/data/label_schema.py`, add the missing class to that source's
`class_map`, and re-run. Do not guess a mapping and move on silently; check the
class semantically (e.g. by rendering a few of its boxes, like was done for
`ultralytics_construction_ppe`'s `"none"` class - see `docs/DATASET_NOTES.md`).

In [ ]:
source_args = " ".join(f"--source {key}={path}" for key, path in source_paths.items())
!python scripts/build_unified_dataset.py {source_args} --out data/unified

import pandas as pd
dist = pd.read_csv("data/unified/distribution.csv")
print(dist.pivot_table(index="unified_class", columns="split", values="instance_count", aggfunc="sum", fill_value=0))

In [ ]:
# Check for duplicate/leakage clusters - pay special attention to any cluster
# that mixes construction_site_safety and ppe_combined_model paths, since
# docs/DATASET_NOTES.md documents a *confirmed* (not hypothetical) image-
# lineage overlap between those two specific sources.
report_path = "data/unified/duplicate_report.txt"
if os.path.exists(report_path):
    with open(report_path) as f:
        report = f.read()
    print(f"Report length: {len(report.splitlines())} lines. First 40 lines:\n")
    print("\n".join(report.splitlines()[:40]))

    cross_source_hits = [
        line for line in report.splitlines()
        if "construction_site_safety__" in line
    ]
    print(f"\n{len(cross_source_hits)} lines reference a construction_site_safety image "
          f"inside some duplicate cluster - inspect data/unified/duplicate_report.txt "
          f"in full and cross-check which clusters also contain a ppe_combined_model__ "
          f"image (the confirmed-overlap pair).")
else:
    print("No duplicate_report.txt written - either no duplicates found, or dedup "
          "was skipped. Re-check the build_unified_dataset.py output above.")

## 5. Train: YOLOv8n baseline, then YOLOv8s comparison

Matches the original baseline author's setup (100 epochs, batch 16) as a
starting point for comparability - see `docs/BASELINE_METRICS.md` for the
numbers this should be compared against. Adjust `epochs`/`batch`/`imgsz` for
your GPU's memory and the time you have available; Kaggle sessions have a
9-hour wall-clock limit, Colab free-tier sessions can disconnect after ~12
hours of idle/active use, so for the full 44k-image merged dataset you may
want to reduce epochs for the first pass and extend later.

In [ ]:
from ultralytics import YOLO

DATA_YAML = "data/unified/data.yaml"

model_n = YOLO("yolov8n.pt")
results_n = model_n.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    project="runs_unified",
    name="yolov8n_unified",
    seed=0,
    plots=True,
)

In [ ]:
# Comparison run: larger backbone (docs/PROJECT_PLAN.md M3 experiment grid).
# Comment this out if you only want the baseline for now - it roughly doubles
# total training time.
model_s = YOLO("yolov8s.pt")
results_s = model_s.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    project="runs_unified",
    name="yolov8s_unified",
    seed=0,
    plots=True,
)

## 6. Evaluation: this repo's own math, not just Ultralytics' defaults

Ultralytics' `model.val()` already gives you mAP/P/R/confusion matrices (like
the ones documented in `docs/BASELINE_METRICS.md` for the original baseline).
This section goes further, using `src/evaluation/` (built and unit-tested in
this repo, 38 tests passing, but never yet run against a real trained model
until now) to compute:

- Per-class precision/recall/F1 from first principles (IoU matching you can
  inspect and trust, not a black box).
- **Calibration**: Expected Calibration Error (ECE) and Brier score - does a
  0.9-confidence detection actually turn out to be correct ~90% of the time?
- **Safety-critical threshold optimization**: for `no_helmet`/`no_vest`/etc.,
  the confidence threshold that guarantees >= 95% recall (tune the target as
  you see fit), and what precision/false-alarm cost that requires - the
  framing from `docs/PROJECT_PLAN.md` M3, made concrete with real numbers.

This will likely take a few minutes (`conf=0.001` deliberately keeps every
low-confidence candidate detection so the threshold analysis has real data to
work with, at the cost of running prediction on more boxes than a normal
inference call would).

In [ ]:
import yaml

from src.data.label_schema import UNIFIED_CLASSES
from src.evaluation.yolo_adapter import evaluate_yolo_val_split

BEST_MODEL_PATH = "runs_unified/yolov8n_unified/weights/best.pt"

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)
class_ids = list(range(len(UNIFIED_CLASSES)))

result, raw = evaluate_yolo_val_split(
    model_path=BEST_MODEL_PATH,
    images_dir="data/unified/val/images",
    labels_dir="data/unified/val/labels",
    class_ids=class_ids,
    iou_threshold=0.5,
    conf=0.001,
)

print(f"{'class':<15} {'precision':>10} {'recall':>10} {'f1':>10} {'tp':>6} {'fp':>6} {'fn':>6}")
for cls_id in class_ids:
    pr = result.per_class[cls_id]
    if pr.tp + pr.fp + pr.fn == 0:
        continue  # class not present in this split at all
    print(f"{UNIFIED_CLASSES[cls_id]:<15} {pr.precision:>10.3f} {pr.recall:>10.3f} {pr.f1:>10.3f} {pr.tp:>6} {pr.fp:>6} {pr.fn:>6}")

micro = result.micro_average()
print(f"\nMicro-average: P={micro.precision:.3f} R={micro.recall:.3f} F1={micro.f1:.3f}")
print(f"Macro F1: {result.macro_f1():.3f}")

In [ ]:
from src.evaluation.calibration import brier_score, expected_calibration_error, reliability_diagram_bins

confidences = raw["confidences"]
is_correct = raw["is_correct"]

ece = expected_calibration_error(confidences, is_correct, n_bins=10)
bs = brier_score(confidences, is_correct)
print(f"Expected Calibration Error (ECE): {ece:.4f}")
print(f"Brier score: {bs:.4f}")

# Reliability diagram (mean confidence vs. empirical accuracy per bin)
import matplotlib.pyplot as plt

bins = reliability_diagram_bins(confidences, is_correct, n_bins=10)
bin_centers = [(b.bin_lower + b.bin_upper) / 2 for b in bins]
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "k--", label="perfect calibration")
plt.plot(bin_centers, [b.empirical_accuracy for b in bins], "o-", label="model")
plt.xlabel("Mean predicted confidence (per bin)")
plt.ylabel("Empirical accuracy (per bin)")
plt.title(f"Reliability diagram (ECE={ece:.3f}, Brier={bs:.3f})")
plt.legend()
plt.savefig("docs_assets_reliability_diagram.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot to docs_assets_reliability_diagram.png - copy this into docs/assets/ "
      "and reference it from docs/EXPERIMENTS.md.")

In [ ]:
from src.evaluation.threshold_optimization import best_f1_operating_point, min_recall_operating_point

# The classes where a missed detection (false negative) is a safety miss, not
# just a minor error - tune this list/target to your own judgment call, but
# make the choice explicit rather than defaulting to "maximize F1" everywhere.
SAFETY_CRITICAL_CLASSES = ["no_helmet", "no_vest", "no_goggles", "no_gloves", "no_boots", "no_mask"]
MIN_RECALL_TARGET = 0.95

class_name_to_id = {name: i for i, name in enumerate(UNIFIED_CLASSES)}

print(f"{'class':<15} {'best-F1 thr':>12} {'best-F1 P/R':>14} | {'safety thr':>12} {'safety P/R':>14}")
for cls_name in SAFETY_CRITICAL_CLASSES:
    cls_id = class_name_to_id[cls_name]
    per_class_data = raw["per_class"].get(cls_id, {"confidences": [], "is_correct": []})
    scores = per_class_data["confidences"]
    y_true = per_class_data["is_correct"]
    if len(set(y_true)) < 2 or len(scores) < 5:
        print(f"{cls_name:<15} insufficient data in this val split ({len(scores)} detections) - skipping")
        continue

    f1_point = best_f1_operating_point(y_true, scores)
    try:
        safety_point = min_recall_operating_point(y_true, scores, min_recall=MIN_RECALL_TARGET)
        safety_str = f"{safety_point.threshold:>12.3f} {safety_point.precision:.3f}/{safety_point.recall:.3f}"
    except ValueError as e:
        safety_str = f"{'N/A':>12} UNACHIEVABLE: {e}"

    print(f"{cls_name:<15} {f1_point.threshold:>12.3f} {f1_point.precision:.3f}/{f1_point.recall:.3f} | {safety_str}")

print(f"\n'best-F1 P/R' and 'safety P/R' columns are precision/recall at that operating point.")
print(f"If any class shows UNACHIEVABLE, that means the model's own recall ceiling is "
      f"below {MIN_RECALL_TARGET} for that class - a model/data problem, not fixable by "
      f"threshold choice alone. Report this honestly rather than picking a lower target "
      f"to make the number look achievable.")

## 7. Getting results back into the repo

Kaggle/Colab GPU sessions are ephemeral - don't lose this run's output. Two
practical options:

**Option A - zip and download, then commit from your own machine:**

```python
!zip -r results_bundle.zip runs_unified/*/weights/best.pt runs_unified/*/results.png \
    runs_unified/*/results.csv runs_unified/*/confusion_matrix.png \
    runs_unified/*/PR_curve.png runs_unified/*/P_curve.png runs_unified/*/R_curve.png \
    runs_unified/*/F1_curve.png data/unified/distribution.csv \
    data/unified/duplicate_report.txt docs_assets_reliability_diagram.png
```
Then, on Kaggle, download `results_bundle.zip` from the notebook's Output
panel; on Colab, `from google.colab import files; files.download("results_bundle.zip")`.
Unzip locally, copy the relevant files into `docs/assets/experiments/`, and
write up the numbers in `docs/EXPERIMENTS.md` (create it following the
structure of `docs/BASELINE_METRICS.md`).

**Option B - push directly from the notebook** (only if you're comfortable
putting a GitHub token in this notebook session - never commit the token
itself):

```python
!git config user.email "you@example.com"
!git config user.name "Your Name"
!git add docs/ data/unified/distribution.csv data/unified/duplicate_report.txt
!git commit -m "Add real M3 training + evaluation results from Kaggle/Colab run"
!git push https://<YOUR_GITHUB_TOKEN>@github.com/A-Kuo/Worker-Safety-PPE-Detection-Model.git HEAD:cursor/ppe-detection-project-plan-7565
```

Either way, do **not** commit `data/unified/` images/labels themselves (it's
already gitignored via `/data/`) or the multi-hundred-MB `runs_unified/`
training run directory - only the small summary artifacts (plots, CSVs,
`best.pt` if you want the trained weights versioned, though consider a
release asset or Git LFS for that instead of a normal commit given its size).